## Identify bias in the model predictions by implementing fairness metrics such as disparate impact and equalized odds without using prebuilt fairness libraries, manually computing bias scores across demographic groups.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_csv("heart.csv")
df = df.drop_duplicates().dropna()

X = df.drop("target", axis=1)
y = df["target"]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

results = X_test.copy()
results["actual"] = y_test
results["predicted"] = predictions

print("Model Accuracy:", model.score(X_test, y_test))

Model Accuracy: 0.7868852459016393


In [3]:
raw_X = df.drop("target", axis=1)

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    raw_X, y, test_size=0.2, random_state=42, stratify=y
)

results["sex"] = X_test_raw["sex"].values
results["actual"] = y_test_raw.values
results["predicted"] = predictions

print(results["sex"].value_counts())

sex
1    41
0    20
Name: count, dtype: int64


In [4]:
group_rates = results.groupby("sex")["predicted"].mean()

print("Positive Prediction Rates:")
print(group_rates)

if len(group_rates) >= 2:
    rates = group_rates.sort_values(ascending=False)
    highest_rate = rates.iloc[0]
    lowest_rate = rates.iloc[-1]

    disparate_impact = (
        lowest_rate / highest_rate if highest_rate != 0 else np.nan
    )

    print("Disparate Impact:", disparate_impact)

    if pd.notna(disparate_impact):
        if disparate_impact < 0.8:
            print("Potential bias detected: Disparate impact is below 0.8.")
        else:
            print("No disparity flagged by the 0.8 screening threshold.")
else:
    print("Not enough demographic groups to calculate disparate impact.")

Positive Prediction Rates:
sex
0    0.900000
1    0.536585
Name: predicted, dtype: float64
Disparate Impact: 0.5962059620596206
Potential bias detected: Disparate impact is below 0.8.


In [5]:
fairness_results = []

for group in sorted(results["sex"].dropna().unique()):
    group_data = results[results["sex"] == group]

    actual = group_data["actual"]
    predicted = group_data["predicted"]

    tp = ((actual == 1) & (predicted == 1)).sum()
    fn = ((actual == 1) & (predicted == 0)).sum()
    fp = ((actual == 0) & (predicted == 1)).sum()
    tn = ((actual == 0) & (predicted == 0)).sum()

    tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

    fairness_results.append({
        "sex": group,
        "TPR": tpr,
        "FPR": fpr,
        "TP": tp,
        "FN": fn,
        "FP": fp,
        "TN": tn
    })

fairness_df = pd.DataFrame(fairness_results)
print(fairness_df)

   sex     TPR       FPR  TP  FN  FP  TN
0    0  1.0000  0.333333  17   0   1   2
1    1  0.8125  0.360000  13   3   9  16


In [6]:
tpr_difference = fairness_df["TPR"].max() - fairness_df["TPR"].min()
fpr_difference = fairness_df["FPR"].max() - fairness_df["FPR"].min()

print("TPR Difference:", tpr_difference)
print("FPR Difference:", fpr_difference)

print("Equalized Odds Difference:", max(tpr_difference, fpr_difference))

TPR Difference: 0.1875
FPR Difference: 0.026666666666666672
Equalized Odds Difference: 0.1875


In [7]:
with open("Model_Bias_and_Fairness_Report.txt", "w") as file:
    file.write("MODEL BIAS AND FAIRNESS ANALYSIS\n\n")
    file.write(f"Model Accuracy: {model.score(X_test, y_test):.4f}\n\n")
    file.write("Positive Prediction Rates:\n")
    file.write(group_rates.to_string())
    file.write("\n\nDisparate Impact:\n")
    file.write(str(disparate_impact))
    file.write("\n\nFairness Metrics by Group:\n")
    file.write(fairness_df.to_string(index=False))
    file.write("\n\nTPR Difference: ")
    file.write(str(tpr_difference))
    file.write("\nFPR Difference: ")
    file.write(str(fpr_difference))
    file.write("\nEqualized Odds Difference: ")
    file.write(str(max(tpr_difference, fpr_difference)))
    file.write("\n\nNote: Metrics indicate disparities, not proof of discrimination.")

print("Report saved successfully.")

Report saved successfully.
